# 01 — Geração do Dataset Simulado

Este notebook gera um dataset simulado de controle de qualidade farmacêutico,
baseado em parâmetros realistas de um processo de compressão de comprimidos.

**Parâmetros simulados:**
- Variáveis de processo: temperatura, umidade, pressão de compressão
- Atributos críticos de qualidade (CQAs): dureza, friabilidade, peso médio, dissolução
- Variável alvo: status do lote (APROVADO / REPROVADO)

In [1]:
import numpy as np
import pandas as pd
import os

np.random.seed(42)
N_LOTES = 500

In [5]:
# --- Variáveis de processo ---
temp_processo     = np.random.normal(loc=25.0, scale=1.5, size=N_LOTES)       # °C
umidade_relativa  = np.random.normal(loc=45.0, scale=5.0, size=N_LOTES)       # %
pressao_compressao = np.random.normal(loc=12.0, scale=1.2, size=N_LOTES)      # kN

# --- CQAs com correlação aos parâmetros de processo ---
dureza_media   = 80 + 2.5 * pressao_compressao + np.random.normal(0, 3, N_LOTES)   # N
friabilidade   = np.clip(0.5 - 0.05 * pressao_compressao + 0.02 * umidade_relativa
                         + np.random.normal(0, 0.1, N_LOTES), 0.1, 3.0)             # %
peso_medio     = np.random.normal(loc=500, scale=5, size=N_LOTES)                   # mg
dissolucao     = np.clip(115.5 - 0.3 * temp_processo - 0.4 * umidade_relativa
                         + np.random.normal(0, 2, N_LOTES), 60, 100)                # %

# --- Metadados ---
datas = pd.date_range(start='2023-01-01', periods=N_LOTES, freq='12h')
turnos = np.random.choice(['A', 'B', 'C'], size=N_LOTES)
lotes  = [f'LOT-{str(i+1).zfill(4)}' for i in range(N_LOTES)]

In [6]:
# --- Critérios de reprovação (baseados em especificações típicas) ---
# Dureza: 70–120 N | Friabilidade: < 1.0% | Dissolução: >= 80% | Peso: 475–525 mg
reprovado = (
    (dureza_media < 70) | (dureza_media > 120) |
    (friabilidade >= 1.0) |
    (dissolucao < 80) |
    (peso_medio < 475) | (peso_medio > 525)
)

status_lote = np.where(reprovado, 'REPROVADO', 'APROVADO')

print(f"Total de lotes: {N_LOTES}")
print(f"Aprovados:  {(status_lote == 'APROVADO').sum()} ({(status_lote == 'APROVADO').mean():.1%})")
print(f"Reprovados: {(status_lote == 'REPROVADO').sum()} ({(status_lote == 'REPROVADO').mean():.1%})")

Total de lotes: 500
Aprovados:  449 (89.8%)
Reprovados: 51 (10.2%)


In [7]:
# --- Montar DataFrame ---
df = pd.DataFrame({
    'lote':               lotes,
    'data_producao':      datas,
    'turno':              turnos,
    'temp_processo':      temp_processo.round(2),
    'umidade_relativa':   umidade_relativa.round(2),
    'pressao_compressao': pressao_compressao.round(2),
    'dureza_media':       dureza_media.round(2),
    'friabilidade':       friabilidade.round(3),
    'peso_medio':         peso_medio.round(2),
    'dissolucao':         dissolucao.round(2),
    'status_lote':        status_lote
})

df.head(10)

,lote,data_producao,turno,temp_processo,umidade_relativa,pressao_compressao,dureza_media,friabilidade,peso_medio,dissolucao,status_lote
0,LOT-0001,2023-01-01 00:00:00,C,25.48,49.99,10.48,109.95,0.979,507.08,89.50,APROVADO
1,LOT-0002,2023-01-01 12:00:00,A,24.98,43.57,12.80,109.67,0.783,503.87,90.19,APROVADO
2,LOT-0003,2023-01-02 00:00:00,A,23.78,48.80,12.81,112.83,0.738,499.21,89.44,APROVADO
3,LOT-0004,2023-01-02 12:00:00,B,26.23,43.54,12.46,112.41,0.607,501.54,93.55,APROVADO
4,LOT-0005,2023-01-03 00:00:00,B,25.34,44.70,10.81,110.59,0.864,497.00,85.77,APROVADO
5,LOT-0006,2023-01-03 12:00:00,C,25.03,47.47,12.30,108.29,0.802,500.70,89.83,APROVADO
6,LOT-0007,2023-01-04 00:00:00,B,23.54,46.45,10.58,109.73,0.963,500.37,94.13,APROVADO
7,LOT-0008,2023-01-04 12:00:00,B,26.96,35.02,11.70,107.60,0.608,504.84,93.97,APROVADO
8,LOT-0009,2023-01-05 00:00:00,C,26.00,39.39,13.24,109.08,0.647,504.84,95.24,APROVADO
9,LOT-0010,2023-01-05 12:00:00,C,24.17,49.65,12.96,114.56,0.683,502.47,87.13,APROVADO


In [8]:
# --- Salvar dataset bruto ---
os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/lotes_qc.csv', index=False)
print("Dataset salvo em: data/raw/lotes_qc.csv")
print(f"Shape: {df.shape}")
df.info()

Dataset salvo em: data/raw/lotes_qc.csv
Shape: (500, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   lote                500 non-null    object        
 1   data_producao       500 non-null    datetime64[ns]
 2   turno               500 non-null    object        
 3   temp_processo       500 non-null    float64       
 4   umidade_relativa    500 non-null    float64       
 5   pressao_compressao  500 non-null    float64       
 6   dureza_media        500 non-null    float64       
 7   friabilidade        500 non-null    float64       
 8   peso_medio          500 non-null    float64       
 9   dissolucao          500 non-null    float64       
 10  status_lote         500 non-null    object        
dtypes: datetime64[ns](1), float64(7), object(3)
memory usage: 43.1+ KB
